# 11 — Comparing Thermodynamic Data Sources: NASA, NIST &amp; Conventional Tables\n
\n
**Thermochemical databases** are the backbone of combustion, propulsion and\n
chemical-kinetics simulations. Yet different data sources — NASA polynomial\n
fits, NIST reference tables and classic JANAF compilations — can yield\n
slightly different values for the same property.\n
\n
This notebook uses **H₂O(g)** as a reference fluid over 300–2500&nbsp;K and\n
compares three independent sources side by side:\n
\n
| Source | Type | How it is accessed |\n
|--------|------|--------------------|\n
| **NASA Polynomials** (Burcat &amp; Ruscic) | Analytical 7-coefficient fit | Direct evaluation of $C_p(T)$, $S(T)$, $H(T)$ |\n
| **NIST Chemistry WebBook** (SRD&nbsp;69) | Tabulated reference data | Online fetch + spline interpolation |\n
| **Conventional Table** (JANAF / steam-table) | Classic printed-table style | Embedded discrete points + spline interpolation |\n
\n
The goals are:\n
\n
1. Reproduce the **NASA polynomial evaluation** by hand — the same math\n
   that ``pyglenn`` performs internally\n
2. Fetch real **NIST data** live from the Chemistry WebBook (with an\n
   embedded fallback for offline use)\n
3. Cross-validate all three sources and quantify **relative discrepancies**\n
4. Benchmark the **computational cost** of each approach\n
\n
> **Author of the package:** Dr. Reginaldo G. Le&atilde;o Jr. — GESESC / IFMG.

## Imports and setup

In [ ]:
import numpy as np\n
import matplotlib.pyplot as plt\n
import time\n
import warnings\n
from typing import Callable, Dict, Tuple\n
from dataclasses import dataclass\n
\n
warnings.filterwarnings(\
)\n
%matplotlib inline\n
plt.rcParams[\
] = (10, 6)\n
plt.rcParams[\
] = True

---\n
\n
## 1. NASA Polynomial Coefficients\n
\n
NASA (Glenn) polynomials express the dimensionless heat capacity as a\n
power series in $T$:\n
\n
$$\n
\\frac{C_p^\\circ}{R} = a_1 + a_2 T + a_3 T^2 + a_4 T^3 + a_5 T^4\n
$$\n
\n
The enthalpy and entropy follow by analytical integration (see Notebooks\n
01 and 02 for the full derivation). Here we use the Burcat &amp; Ruscic\n
coefficients for **H₂O(g)** — two temperature intervals: low\n
(200–1000&nbsp;K) and high (1000–5000&nbsp;K).

In [ ]:
# Coefficients from Burcat & Ruscic database (7-coefficient NASA format)\n
# H₂O(g) — low interval (200–1000 K) and high interval (1000–5000 K)\n
\n
NASA_H2O_LOW = np.array([\n
     3.38684237E+00,  3.47498246E-03, -6.35469633E-06,\n
     6.96858128E-09, -2.50658848E-12, -3.02937267E+04,  2.59023255E+00\n
])\n
\n
NASA_H2O_HIGH = np.array([\n
     2.56056053E+00,  1.01681151E-03,  1.26487976E-07,\n
    -1.44606925E-10,  1.85788218E-14, -3.00875095E+04,  8.31820890E+00\n
])\n
\n
T_MID = 1000.0          # Transition temperature between coefficient sets\n
R_UNIV = 8.314462618    # J/(mol·K)\n
\n
def nasa_coefficients(T):\n
    \
\
\
\n
    return NASA_H2O_LOW if T < T_MID else NASA_H2O_HIGH\n
\n
def cp_nasa(T):\n
    \
\
\
\n
    a = nasa_coefficients(T)\n
    return R_UNIV * (a[0] + a[1]*T + a[2]*T**2 + a[3]*T**3 + a[4]*T**4)\n
\n
def h_nasa(T):\n
    \
\
\
\n
    a = nasa_coefficients(T)\n
    T_ref = 298.15\n
    a_ref = nasa_coefficients(T_ref)\n
\n
    def h_integral(a_arr, temp):\n
        return R_UNIV * temp * (\n
            a_arr[0] + a_arr[1]*temp/2 + a_arr[2]*temp**2/3\n
            + a_arr[3]*temp**3/4 + a_arr[4]*temp**4/5 + a_arr[5]/temp\n
        )\n
\n
    return h_integral(a, T) - h_integral(a_ref, T_ref)\n
\n
def s_nasa(T):\n
    \
\
\
\n
    a = nasa_coefficients(T)\n
    T_ref = 298.15\n
    a_ref = nasa_coefficients(T_ref)\n
\n
    def s_integral(a_arr, temp):\n
        return R_UNIV * (\n
            a_arr[0]*np.log(temp) + a_arr[1]*temp + a_arr[2]*temp**2/2\n
            + a_arr[3]*temp**3/3 + a_arr[4]*temp**4/4 + a_arr[6]\n
        )\n
\n
    return s_integral(a, T) - s_integral(a_ref, T_ref)\n
\n
print(\
)\n
print(f\
print(f\
)

---\n
\n
## 2. NIST Chemistry WebBook\n
\n
The [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/) (SRD&nbsp;69)\n
provides tabulated thermodynamic data — $C_p^\\circ$, $S^\\circ$, and\n
$[H^\\circ - H^\\circ(298.15)]$ — at discrete temperature points for\n
thousands of compounds.\n
\n
We fetch the ideal-gas data for H₂O directly from the WebBook using an HTTP\n
request. When the network is unavailable, the function falls back to a\n
pre-embedded dataset so the notebook remains **fully reproducible offline**.

In [ ]:
NIST_URL = (\n
    \